In [ ]:
#| default_exp activations

# Load deps

In [ ]:
# !pip install -q torcheval
# # or
# !uv add torcheval

In [ ]:
# # if src modules imported
# from google.colab import drive
# drive.mount('/content/drive')
# import sys
# from pathlib import Path
# app_path = '/content/drive/MyDrive/Projects/miniSD'
# sys.path.append(app_path)

In [ ]:
#|export
from typing import Callable, Optional
import random,math,torch
import numpy as np
import matplotlib.pyplot as plt
import fastcore.all as fc
from functools import partial
from src.datasets import inplace, DataLoaders
from src.learner import (
    MetricsCB, TrainCB, DeviceCB,
    ProgressCB, Learner, to_cpu, Callback
)
from src.utils import store_attr
from src.datasets import get_grid, show_image

In [ ]:
import torch.nn.functional as F
import matplotlib as mpl
from torch import tensor,nn
import torchvision.transforms.functional as TF
from datasets import load_dataset
from torcheval.metrics import MulticlassAccuracy

In [ ]:
#|export --> utils
def set_seed(seed, deterministic=False):
    torch.use_deterministic_algorithms(deterministic)
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)

# Config

In [ ]:
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
mpl.rcParams['figure.constrained_layout.use'] = True
dataset_name = "fashion_mnist"
x,y = 'image','label'
bs = 1024
num_dl_workers = 0
# using single worker in colab. increase #workers in prod


# Load data

In [ ]:
@inplace
def transformi(b): b[x] = [TF.to_tensor(o) for o in b[x]]

dsd = load_dataset(dataset_name)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(
    tds, bs, num_workers=num_dl_workers
)
dt = dls.train

# Baseline

- We want to train quickly, so that means training at a high learning rate. Why?
  - training with a big `lr` means we can train stably which is highly desired
  - training with a high `lr` prevents overfitting since:
    - it prevents getting stick in local minima
    - it means looking at the same batch less often

In [ ]:
def conv(ni, nf, ks=3, act=True):
    res = nn.Conv2d(ni, nf, stride=2, kernel_size=ks, padding=ks//2)
    if act: res = nn.Sequential(res, nn.ReLU())
    return res

def cnn_layers():
    return [
        conv(1 ,8, ks=5),        #14x14
        conv(8 ,16),             #7x7
        conv(16,32),             #4x4
        conv(32,64),             #2x2
        conv(64,10, act=False),  #1x1
        nn.Flatten()]

def fit(model, epochs=1, cbs=[]):
    learn = Learner(model, dls, loss_func=F.cross_entropy, lr=0.6, cbs=cbs)
    learn.fit(epochs)
    return learn

In [ ]:
metrics = MetricsCB(accuracy=MulticlassAccuracy())
cbs = [TrainCB(), DeviceCB(), metrics, ProgressCB(plot=True)]

In [ ]:
# set_seed(1)
# learn = fit(model = nn.Sequential(*cnn_layers()), cbs=cbs)

# Hooks

## Manual insertion

In [ ]:
class SequentialModel(nn.Module):
    def __init__(self, *layers):
        super().__init__()
        self.layers = nn.ModuleList(layers)
        self.act_means = [[] for _ in layers]
        self.act_stds  = [[] for _ in layers]

    def __call__(self, x):
        for i,l in enumerate(self.layers):
            x = l(x)
            self.act_means[i].append(to_cpu(x).mean())
            self.act_stds [i].append(to_cpu(x).std ())
        return x

    def __iter__(self): return iter(self.layers)

In [ ]:
set_seed(1)
model = SequentialModel(*cnn_layers())
learn = fit(model, cbs=cbs)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(5,6), sharex=True, sharey=True)
axs[0].set_title('layer mean')
axs[1].set_title('layer std')
for l in model.act_means:
  axs[0].plot(l)
for l in model.act_stds:
  axs[1].plot(l)

for ax in axs:
  ax.legend(range(5));
  ax.set_xlabel('batch');

## Pytorch hooks

- Hooks are PyTorch object you can add to any nn.Module.
- A hook will be called when a layer, it is registered to, is executed during the forward pass (forward hook) or the backward pass (backward hook).
- Hooks don't require us to rewrite the model.
- A hook is attached to a layer, and needs to have a function that takes three arguments
  - module
  - input
  - output

In [ ]:
set_seed(1)
model = nn.Sequential(*cnn_layers())

act_means = [[] for _ in model]
act_stds  = [[] for _ in model]

def append_stats(i, mod, inp, outp):
    act_means[i].append(to_cpu(outp).mean())
    act_stds [i].append(to_cpu(outp).std())

for i,m in enumerate(model): m.register_forward_hook(partial(append_stats, i))

fit(model, cbs=cbs)

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(5,6), sharex=True, sharey=True)
axs[0].set_title('layer mean')
axs[1].set_title('layer std')
for m,s in zip(act_means, act_stds):
  axs[0].plot(m)
  axs[1].plot(s)
for ax in axs:
  ax.legend(range(5));
  ax.set_xlabel('batch');

## Hook class

- We can refactor this in a Hook class.
- It's very important to **remove** the hooks when they are deleted
- **otherwise** there will be references kept and the memory won't be properly released when your model is deleted.

In [ ]:
#| export
class Hook():
    def __init__(self, m, f): self.hook = m.register_forward_hook(partial(f, self))
    def remove(self): self.hook.remove()
    def __del__(self): self.remove()

In [ ]:
def append_stats(hook, mod, inp, outp):
    if not hasattr(hook,'stats'): hook.stats = ([],[])
    acts = to_cpu(outp)
    hook.stats[0].append(acts.mean())
    hook.stats[1].append(acts.std())

In [ ]:
set_seed(1)
model = nn.Sequential(*cnn_layers())
hooks = [Hook(l, append_stats) for l in model[:5].children()]
learn = fit(model, cbs=cbs)

## A Hooks class

In [ ]:
#| export
class Hooks(list):
    def __init__(self, ms, f): super().__init__([Hook(m, f) for m in ms])
    def __enter__(self, *args): return self
    def __exit__ (self, *args): self.remove()
    def __del__(self): self.remove()
    def __delitem__(self, i):
        self[i].remove()
        super().__delitem__(i)
    def remove(self):
        for h in self: h.remove()

In [ ]:
set_seed(1)
model = nn.Sequential(*cnn_layers())

fig, axs = plt.subplots(2, 1, figsize=(5,6), sharex=True, sharey=True)
axs[0].set_title('layer mean')
axs[1].set_title('layer std')

with Hooks(model, append_stats) as hooks:
  fit(model, cbs=cbs)
  for h in hooks:
    for i,ax in enumerate(axs):
      ax.plot(h.stats[i])
      ax.set_xlabel('batch');
plt.legend(range(len(model)));

## HooksCallback

In [ ]:
#| export
class HooksCallback(Callback):
    def __init__(
        self,
        hookfunc: Callable,
        mod_filter: Callable = lambda _:True,
        on_train: bool = True,
        on_valid: bool = False,
        mods: Optional[list] = None
    ):
        store_attr()
        super().__init__()

    def before_fit(self, learn):
        if self.mods: mods=self.mods
        else: mods = [mod for mod in learn.model.modules() if self.mod_filter(mod)]
        self.hooks = Hooks(mods, partial(self._hookfunc, learn))

    def _hookfunc(self, learn, *args, **kwargs):
        if (self.on_train and learn.training) or (self.on_valid and not learn.training):
          self.hookfunc(*args, **kwargs)

    def after_fit(self, learn): self.hooks.remove()
    def __iter__(self): return iter(self.hooks)
    def __len__(self): return len(self.hooks)

In [ ]:
hc = HooksCallback(
    append_stats, mod_filter=lambda mod: isinstance(mod, nn.Conv2d)
)
set_seed(1)
model = nn.Sequential(*cnn_layers())
fit(model, cbs=cbs+[hc]);

fig,axs = plt.subplots(1,2, figsize=(10,4))
for h in hc:
    for i in 0,1: axs[i].plot(h.stats[i])
plt.legend(range(6));

# Histograms

In [ ]:
#| export
def append_stats(hook, mod, inp, outp):
    if not hasattr(hook,'stats'): hook.stats = ([],[],[])
    acts = to_cpu(outp)
    hook.stats[0].append(acts.mean())
    hook.stats[1].append(acts.std())
    hook.stats[2].append(acts.abs().histc(40,0,10))

def get_hist(h): return torch.stack(h.stats[2]).t().float().log1p()
def get_min(h):
    h1 = torch.stack(h.stats[2]).t().float()
    return h1[0]/h1.sum(0)

In [ ]:
set_seed(1)
model = nn.Sequential(*cnn_layers())
hc = HooksCallback(
    append_stats,
    mod_filter=lambda mod: isinstance(mod, nn.Conv2d)
)
fit(model, cbs=cbs+[hc]);

In [ ]:
fig,axes = get_grid(len(hc), figsize=(9,4))
for ax,h in zip(axes.flat, hc):
    show_image(get_hist(h), ax, origin='lower')

## How to interpret the histogram?

- each column of pixels is showing the color-coded distribution of
  - absoulte values of activations for than specific batch/step
- values start from zero (at the bottom) and increase toward the top (the x axis is binned)
  - lighter colors show higher frequencies and darker colors show lower frequencies
- the ideal plot should be a rectangular pattern demonstrating a distribution
  - that is derived from a normal distriution after getting absolute values
  - min and max should not change dramatically
  - not having a lot of zero or near-zero values since these are basically inactive nodes
  - the most frequent bins are those at the middle of the plot (neither too large nor too small)
  - and the bins get less frequent (darker) as we get further away from this region

- If the plot shows
  - repetitive rise and fall patterns at the start of the training (ealy steps)
  - a large ratio of near-zero activaions
    - this model is most lokely not going to recover
    - beacuse a lot of activations have diverged
    - just reset the training with modified configurations

## near-zero ratio plot
- should be (ideally)
  - not so large
  - steady

In [ ]:
fig,axes = get_grid(len(hc), figsize=(11,5))
for ax,h in zip(axes.flatten(), hc):
    ax.plot(get_min(h))
    ax.set_ylim(0,1)

# ActivationStats

In [ ]:
#|export
class ActivationStats(HooksCallback):
    def __init__(
        self,
        mod_filter: Callable = lambda _:True,
    ): super().__init__(append_stats, mod_filter)

    def color_dim(self, figsize=(11,5)):
        fig,axes = get_grid(len(self), figsize=figsize)
        for ax,h in zip(axes.flat, self):
            show_image(get_hist(h), ax, origin='lower')

    def dead_chart(self, figsize=(11,5)):
        fig,axes = get_grid(len(self), figsize=figsize)
        for ax,h in zip(axes.flatten(), self):
            ax.plot(get_min(h))
            ax.set_ylim(0,1)

    def plot_stats(self, figsize=(10,4)):
        fig,axs = plt.subplots(1,2, figsize=figsize)
        for h in self:
            for i in 0,1: axs[i].plot(h.stats[i])
        axs[0].set_title('Means')
        axs[1].set_title('Stdevs')
        plt.legend(range(len(self)))

In [ ]:
astats = ActivationStats(fc.risinstance(nn.Conv2d))
set_seed(1)
model = nn.Sequential(*cnn_layers())
fit(model, cbs=cbs+[astats]);

In [ ]:
astats.color_dim()

In [ ]:
astats.dead_chart()

In [ ]:
astats.plot_stats()